In [ ]:

# =============================================================================
# LSTM V2 FORECASTING — CELL 1
# ENVIRONMENT AND FROZEN DATASET VERIFICATION
#
# NO MODEL TRAINING
# NO OFFICIAL VALIDATION OR TEST LOADING
# =============================================================================

from pathlib import Path
import hashlib
import json
import random
import platform

import numpy as np
import pandas as pd
import torch

from google.colab import drive

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

SEED = 20261022

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

DATA_DIR = (
    PROJECT_ROOT
    / "Synthetic_NAF_v2"
    / "release_candidate_corrected"
)

FREEZE_DIR = (
    PROJECT_ROOT
    / "Synthetic_NAF_v2"
    / "FINAL_RELEASE_FREEZE"
)

MODEL_ROOT = (
    PROJECT_ROOT
    / "Synthetic_NAF_v2"
    / "models"
    / "LSTM_V2_Horizon"
)

TRAIN_FILE = DATA_DIR / "windows_W10_TRAIN.npz"

META_FILE = (
    DATA_DIR
    / "episode_metadata_DO_NOT_USE_AS_PREDICTORS.csv"
)

CONTRACT_FILE = (
    DATA_DIR
    / "FEATURE_AND_DATASET_CONTRACT.json"
)

MANIFEST_FILE = (
    FREEZE_DIR
    / "Synthetic_NAF_v2_RELEASE_MANIFEST.json"
)

LOCK_FILE = (
    FREEZE_DIR
    / "Synthetic_NAF_v2_FREEZE_LOCK.json"
)

HASH_FILE = DATA_DIR / "SHA256SUMS.json"


def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)
    return h.hexdigest()


def canonical_hash(value):
    payload = json.dumps(
        value,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False
    ).encode("utf-8")

    return hashlib.sha256(payload).hexdigest()


for path in [
    TRAIN_FILE,
    META_FILE,
    CONTRACT_FILE,
    MANIFEST_FILE,
    LOCK_FILE,
    HASH_FILE
]:
    if not path.is_file():
        raise FileNotFoundError(path)

CONTRACT = read_json(CONTRACT_FILE)
MANIFEST = read_json(MANIFEST_FILE)
FREEZE_LOCK = read_json(LOCK_FILE)

assert MANIFEST["release_version"] == "2.0.0"
assert MANIFEST["release_status"] == (
    "FROZEN_CONTROLLED_SYNTHETIC_BENCHMARK"
)

assert (
    canonical_hash(MANIFEST)
    == FREEZE_LOCK["manifest_sha256_canonical"]
)

RECORDED_HASHES = read_json(HASH_FILE)

for filename, expected in RECORDED_HASHES.items():
    actual = sha256_file(DATA_DIR / filename)

    if actual != expected:
        raise RuntimeError(
            f"Dataset integrity failure: {filename}"
        )

for filename, expected in (
    MANIFEST["data_integrity"]["file_sha256"].items()
):
    actual = sha256_file(DATA_DIR / filename)

    if actual != expected:
        raise RuntimeError(
            f"Manifest integrity failure: {filename}"
        )

FEATURES = list(CONTRACT["features"])
FAMILIES = list(CONTRACT["attack_families"])

HORIZONS = [5, 10, 15]
HORIZON_TO_COLUMN = {5: 0, 10: 1, 15: 2}

assert len(FEATURES) == 14
assert len(FAMILIES) == 5

assert CONTRACT["history_minutes"] == 10
assert CONTRACT["forecast_horizons"] == HORIZONS

print("=" * 90)
print("LSTM V2 — FROZEN RELEASE VERIFIED")
print("=" * 90)
print("Release version:", MANIFEST["release_version"])
print("Torch version:", torch.__version__)
print("Python:", platform.python_version())
print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

print("TRAIN source:", TRAIN_FILE)
print("Predictors:", len(FEATURES))
print("Horizons:", HORIZONS)
print("Dataset SHA256 verification: PASS")
print("Model training: NOT STARTED")
print("Official VALIDATION / TEST: NOT LOADED")


In [ ]:

# =============================================================================
# LSTM V2 — CELL 2
# LOAD AND VERIFY TRAIN-ONLY TEMPORAL SEQUENCES
#
# NO MODEL TRAINING
# =============================================================================

with np.load(TRAIN_FILE, allow_pickle=False) as pack:
    X_RAW = pack["X"].astype(np.float32)

    Y_HORIZONS = pack["y_horizons"].astype(np.float32)
    ONSET_CLASS = pack["onset_class"].astype(np.int16)
    FAMILY_CLASS = pack["family_class"].astype(np.int8)

    EPISODE_ID = (
        pd.Series(pack["episode_id"])
        .astype(str)
        .to_numpy()
    )

    MINUTE = pack["minute"].astype(np.int16)

META = pd.read_csv(META_FILE)
META["Episode_ID"] = META["Episode_ID"].astype(str)

TRAIN_META = (
    META.loc[META["Split"] == "TRAIN"]
    .set_index("Episode_ID")
)

assert TRAIN_META.index.is_unique

assert X_RAW.shape == (88186, 10, 14)
assert Y_HORIZONS.shape == (len(X_RAW), 3)

assert np.isfinite(X_RAW).all()
assert (X_RAW >= 0).all()

assert len(np.unique(EPISODE_ID)) == 1400
assert set(EPISODE_ID) == set(TRAIN_META.index)

assert np.array_equal(
    Y_HORIZONS[:, 0],
    ((ONSET_CLASS >= 1) & (ONSET_CLASS <= 5))
)

assert np.array_equal(
    Y_HORIZONS[:, 1],
    ((ONSET_CLASS >= 1) & (ONSET_CLASS <= 10))
)

assert np.array_equal(
    Y_HORIZONS[:, 2],
    ((ONSET_CLASS >= 1) & (ONSET_CLASS <= 15))
)

assert np.all(
    Y_HORIZONS[:, 0] <= Y_HORIZONS[:, 1]
)

assert np.all(
    Y_HORIZONS[:, 1] <= Y_HORIZONS[:, 2]
)

assert np.all(
    (ONSET_CLASS == 0) == (FAMILY_CLASS == 0)
)

SOURCE = (
    TRAIN_META.loc[EPISODE_ID, "Source"]
    .astype(str)
    .to_numpy()
)

CAL_GROUP = (
    TRAIN_META.loc[
        EPISODE_ID,
        "Calibration_Group_ID"
    ]
    .astype(str)
    .to_numpy()
)

CV_GROUPS = np.array([
    f"{src}::{grp}"
    for src, grp in zip(SOURCE, CAL_GROUP)
])

assert len(np.unique(CV_GROUPS)) == 35

# The episode ID and minute are audit identifiers only.
# Neither is supplied to the neural network.

print("=" * 90)
print("LSTM V2 — TRAIN ARRAYS VERIFIED")
print("=" * 90)

print("Sequence shape:", X_RAW.shape)
print("Targets:", Y_HORIZONS.shape)
print("Unique episodes:", len(np.unique(EPISODE_ID)))
print("Calibration groups:", len(np.unique(CV_GROUPS)))

for horizon, col in HORIZON_TO_COLUMN.items():
    print(
        f"H{horizon} positive prevalence:",
        f"{Y_HORIZONS[:, col].mean():.6f}"
    )

print("\nPredictor example:")
print("Oldest observed minute:", X_RAW[0, 0, :3])
print("Newest observed minute:", X_RAW[0, -1, :3])

print("\nModel training: NOT STARTED")


In [ ]:

# =============================================================================
# LSTM V2 — CELL 3
# FEATURE-WISE PREPROCESSING AND ROBUST SCALER
#
# No future observations.
# No validation/test fitting.
# No model training.
# =============================================================================

class TemporalRobustScaler:

    def __init__(self, clip_value=8.0):
        self.clip_value = float(clip_value)
        self.median_ = None
        self.scale_ = None

    def fit(self, X):
        assert X.ndim == 3
        assert X.shape[-1] == 14

        # All values are nonnegative from the frozen contract.
        z = np.log1p(X.astype(np.float32))

        # Feature-wise statistics across windows and timesteps.
        # Safe for the supplied training subset, but ONLY
        # use the fold-fitting partition during CV.
        flat = z.reshape(-1, z.shape[-1])

        q25, q50, q75 = np.quantile(
            flat,
            [0.25, 0.50, 0.75],
            axis=0
        )

        self.median_ = q50.astype(np.float32)

        self.scale_ = np.maximum(
            (q75 - q25).astype(np.float32),
            0.05
        )

        return self

    def transform(self, X):
        if self.median_ is None:
            raise RuntimeError("Scaler has not been fitted.")

        z = np.log1p(X.astype(np.float32))

        z = (
            z - self.median_[None, None, :]
        ) / self.scale_[None, None, :]

        z = np.clip(
            z,
            -self.clip_value,
            self.clip_value
        )

        return np.ascontiguousarray(
            z,
            dtype=np.float32
        )

    def state_dict(self):
        if self.median_ is None:
            raise RuntimeError("Scaler has not been fitted.")

        return {
            "median": self.median_.tolist(),
            "scale": self.scale_.tolist(),
            "clip_value": self.clip_value,
            "feature_names": FEATURES
        }


# Preprocessing smoke test on TRAIN data.
# This full-TRAIN scaler is for future final training only.
# Cross-validation must fit a new scaler per fitting fold.

FINAL_TRAIN_SCALER = TemporalRobustScaler(
    clip_value=8.0
).fit(X_RAW)

X_SCALED_PREVIEW = FINAL_TRAIN_SCALER.transform(
    X_RAW[:128]
)

assert X_SCALED_PREVIEW.shape == (128, 10, 14)
assert np.isfinite(X_SCALED_PREVIEW).all()

# Verify a single observation receives identical
# transformation regardless of its timestep position.
same_value = np.repeat(
    X_RAW[:1, :1, :],
    10,
    axis=1
)

same_value_scaled = FINAL_TRAIN_SCALER.transform(
    same_value
)

assert np.allclose(
    same_value_scaled[:, 0, :],
    same_value_scaled[:, 9, :]
)

print("=" * 90)
print("LSTM V2 — PREPROCESSING VERIFIED")
print("=" * 90)

print("Log transform: log1p")
print("Scaling: per-feature median / IQR")
print("Clip range: [-8, +8]")
print("Full-TRAIN preview shape:", X_SCALED_PREVIEW.shape)

print("\nMedian parameters:", FINAL_TRAIN_SCALER.median_)
print("IQR scales:", FINAL_TRAIN_SCALER.scale_)

print("\nFold-level scalers: WILL BE FIT PER FOLD")
print("No VALIDATION or TEST preprocessing performed")
print("Model training: NOT STARTED")


In [ ]:

# =============================================================================
# LSTM V2 — CELL 5
# DEFINE SINGLE-LAYER MONOTONIC THREE-HORIZON LSTM
#
# INPUT:
#   (batch, 10, 14)
#
# OUTPUT:
#   Three cumulative probabilities:
#       P(attack within 5 minutes)
#       P(attack within 10 minutes)
#       P(attack within 15 minutes)
#
# MATHEMATICAL CONSTRAINT:
#   0 <= P(H5) <= P(H10) <= P(H15) <= 1
#
# NO TRAINING
# =============================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F


class MonotonicHorizonHead(nn.Module):

    def __init__(self, input_dim):
        super().__init__()

        self.linear = nn.Linear(
            input_dim,
            3
        )

    def forward(self, hidden):

        # Conditional event probabilities over intervals.
        conditional_logits = self.linear(hidden)

        q = torch.sigmoid(conditional_logits)

        # Numerically stable cumulative survival calculation.
        survival = torch.cumprod(
            1.0 - q,
            dim=-1
        )

        cumulative = 1.0 - survival

        return cumulative, conditional_logits


class LSTMHorizonForecaster(nn.Module):

    def __init__(
        self,
        input_dim=14,
        hidden_dim=64,
        num_layers=2,
        dropout=0.2,
        dense_dim=32
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=False,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.projection = nn.Sequential(
            nn.Linear(hidden_dim, dense_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.head = MonotonicHorizonHead(
            input_dim=dense_dim
        )

    def forward(self, x):

        if x.ndim != 3:
            raise ValueError(
                f"Expected 3D input, got {x.shape}"
            )

        if x.shape[1:] != (10, 14):
            raise ValueError(
                f"Expected (batch,10,14), got {x.shape}"
            )

        output, (h_n, c_n) = self.lstm(x)

        # Last-layer final hidden state, corresponding
        # to the newest observed minute of each window.
        final_hidden = h_n[-1]

        representation = self.projection(
            final_hidden
        )

        probabilities, conditional_logits = self.head(
            representation
        )

        return {
            "probabilities": probabilities,
            "conditional_logits": conditional_logits
        }


MODEL_CONFIG = {
    "input_dim": 14,
    "hidden_dim": 64,
    "num_layers": 1,
    "dropout": 0.1,
    "dense_dim": 32
}

model = LSTMHorizonForecaster(
    **MODEL_CONFIG
).to(DEVICE)

parameter_count = sum(
    p.numel()
    for p in model.parameters()
)

print("=" * 90)
print("LSTM V2 — ARCHITECTURE DEFINED")
print("=" * 90)

print(model)

print("\nTrainable parameters:", parameter_count)
print("Input: (batch, 10, 14)")
print("Output: (batch, 3)")
print("Horizon probability constraint: MONOTONIC")
print("Model weights: RANDOMLY INITIALIZED")
print("Training: NOT STARTED")


In [ ]:
# CELL 5 — FIT THE SINGLE-LAYER BASELINE (TRAIN ONLY)
# Fixed design, no search, no validation fitting, no TEST reads.
# Output folder is intentionally separate from existing trained models.
from torch.utils.data import DataLoader, TensorDataset
import os, time

SINGLE_DIR = PROJECT_ROOT / "Synthetic_NAF_v2" / "models" / "LSTM_V2_SingleLayer"
SINGLE_MODEL_PATH = SINGLE_DIR / "LSTM_V2_SINGLE_LAYER_STATE.pt"
SINGLE_SCALER_PATH = SINGLE_DIR / "LSTM_V2_SINGLE_LAYER_SCALER.json"
SINGLE_RECORD_PATH = SINGLE_DIR / "LSTM_V2_SINGLE_LAYER_TRAIN_RECORD.json"

# Refuse to overwrite any existing baseline artifact. Resuming requires a separate procedure.
if any(p.exists() for p in [SINGLE_MODEL_PATH, SINGLE_SCALER_PATH, SINGLE_RECORD_PATH]):
    raise FileExistsError(f"Existing single-layer artifact found in {SINGLE_DIR}. Stop instead of retraining/overwriting.")

assert MODEL_CONFIG["num_layers"] == 1, "This notebook must train a ONE-layer LSTM."
assert X_RAW.shape == (88186, 10, 14)
assert Y_HORIZONS.shape == (88186, 3)

EPOCHS = 12       # fixed baseline choice, NOT VAL tuned
BATCH_SIZE = 512
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.001
GRAD_CLIP = 1.0

# A new model; do not reload or modify the stacked-LSTM checkpoint.
model = LSTMHorizonForecaster(**MODEL_CONFIG).to(DEVICE)
train_x = FINAL_TRAIN_SCALER.transform(X_RAW)
train_y = np.asarray(Y_HORIZONS, dtype=np.float32)
loader = DataLoader(
    TensorDataset(torch.from_numpy(train_x), torch.from_numpy(train_y)),
    batch_size=BATCH_SIZE, shuffle=True, drop_last=False,
    num_workers=0, pin_memory=(DEVICE.type == 'cuda')
)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

print(f'Training 1-layer LSTM on {len(train_x):,} TRAIN windows, GPU={DEVICE.type=="cuda"}')
loss_history=[]
for ep in range(1, EPOCHS+1):
    model.train()
    total, count = 0.0, 0
    start = time.time()
    for xb,yb in loader:
        xb=xb.to(DEVICE, non_blocking=True)
        yb=yb.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        out=model(xb)['probabilities']
        loss=torch.nn.functional.binary_cross_entropy(out.clamp(1e-6,1-1e-6),yb)
        if not torch.isfinite(loss):
            raise FloatingPointError(f'Non-finite training loss at epoch {ep}')
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        total += loss.item()*len(xb)
        count += len(xb)
    avg=total/count
    loss_history.append(avg)
    print(f'Epoch {ep:02d}/{EPOCHS}: TRAIN BCE={avg:.6f}, seconds={time.time()-start:.1f}')

model.eval()
SINGLE_DIR.mkdir(parents=True,exist_ok=True)
# Save only AFTER successful training; all artifacts live in the isolated new folder.
tmp_model = SINGLE_MODEL_PATH.with_suffix('.tmp')
torch.save({k:v.detach().cpu() for k,v in model.state_dict().items()},tmp_model)
os.replace(tmp_model,SINGLE_MODEL_PATH)

scaler_state=FINAL_TRAIN_SCALER.state_dict()
SINGLE_SCALER_PATH.write_text(json.dumps(scaler_state,indent=2),encoding='utf-8')
record={
 'model_name':'LSTM_V2_SingleLayer',
 'architecture':MODEL_CONFIG,
 'training_rows':len(train_x),
 'training_unique_episodes':int(len(np.unique(EPISODE_ID))),
 'training_epochs':EPOCHS,
 'batch_size':BATCH_SIZE,
 'learning_rate':LEARNING_RATE,
 'weight_decay':WEIGHT_DECAY,
 'loss_function':'mean_unweighted_BCE_on_three_cumulative_horizons',
 'final_train_bce':loss_history[-1],
 'train_loss_by_epoch':loss_history,
 'data_manifest_sha256':FREEZE_LOCK['manifest_sha256_canonical'],
 'model_sha256':sha256_file(SINGLE_MODEL_PATH),
 'scaler_sha256':sha256_file(SINGLE_SCALER_PATH),
 'official_validation_used_for_selection':False,
 'official_test_accessed':False,
 'note':'Fixed baseline; no hyperparameter tuning or early stopping on VAL.'
}
SINGLE_RECORD_PATH.write_text(json.dumps(record,indent=2),encoding='utf-8')
print('\nSINGLE-LAYER TRAINING COMPLETE')
print('Checkpoint:', SINGLE_MODEL_PATH)
print('Model SHA256:',record['model_sha256'])
print('Full TRAIN scaler:',SINGLE_SCALER_PATH)
print('Official TEST accessed: NO')


In [ ]:
# CELL 6 — OFFICIAL VALIDATION REPORT (READS VAL ONLY, NO MODEL SELECTION)
# This is a first-pass baseline confirmation, not the final locked test protocol.
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss
from torch.utils.data import DataLoader,TensorDataset

VAL_PATH = DATA_DIR / "windows_W10_VALIDATION.npz"
assert VAL_PATH.exists(),VAL_PATH
assert SINGLE_MODEL_PATH.exists(), 'Run Cell 5 to train the missing model first.'

with np.load(VAL_PATH,allow_pickle=False) as z:
    xv=z['X'].astype(np.float32)
    yv=z['y_horizons'].astype(np.float32)
    epid=z['episode_id'].astype(str)
    minute=z['minute']
assert xv.shape==(18839,10,14)
assert yv.shape==(18839,3)

# Reconstruct checkpoint from its own saved model and TRAIN-fitted scaler.
sc=json.loads(SINGLE_SCALER_PATH.read_text())
med=np.asarray(sc['median'],dtype=np.float32)
scale=np.asarray(sc['scale'],dtype=np.float32)
assert med.shape==(14,) and scale.shape==(14,)
xv_scaled=np.clip((np.log1p(xv)-med[None,None,:])/scale[None,None,:],-float(sc['clip_value']),float(sc['clip_value'])).astype(np.float32)

report=json.loads(SINGLE_RECORD_PATH.read_text())
assert report['model_sha256']==sha256_file(SINGLE_MODEL_PATH)
model_val=LSTMHorizonForecaster(**report['architecture']).to(DEVICE)
model_val.load_state_dict(torch.load(SINGLE_MODEL_PATH,map_location='cpu',weights_only=True),strict=True)
model_val.eval()
prob=[]
with torch.inference_mode():
    for (xb,) in DataLoader(TensorDataset(torch.from_numpy(xv_scaled)),batch_size=1024,shuffle=False,num_workers=0):
        prob.append(model_val(xb.to(DEVICE))['probabilities'].cpu().numpy())
prob=np.concatenate(prob)
assert prob.shape==(18839,3)
assert np.isfinite(prob).all()
assert (prob>=0).all() and (prob<=1).all()
assert (prob[:,0]<=prob[:,1]+1e-7).all()
assert (prob[:,1]<=prob[:,2]+1e-7).all()

metrics={}
for j,h in enumerate([5,10,15]):
    metrics[f'H{h}']={
      'average_precision':float(average_precision_score(yv[:,j],prob[:,j])),
      'roc_auc':float(roc_auc_score(yv[:,j],prob[:,j])),
      'brier':float(brier_score_loss(yv[:,j],prob[:,j])),
      'prevalence':float(yv[:,j].mean())
    }
    print(f'H{h} | AP={metrics[f"H{h}"]["average_precision"]:.6f} | AUROC={metrics[f"H{h}"]["roc_auc"]:.6f} | Brier={metrics[f"H{h}"]["brier"]:.6f}')

val_dir=SINGLE_DIR/'official_validation'
if val_dir.exists() and any(val_dir.iterdir()):
    raise FileExistsError(f'Validation already exists: {val_dir}; refusing to overwrite')
val_dir.mkdir(parents=True,exist_ok=True)
val_csv=pd.DataFrame({'Episode_ID':epid,'Minute':minute})
for j,h in enumerate([5,10,15]):
    val_csv[f'Y_H{h}']=yv[:,j].astype(int)
    val_csv[f'P_H{h}']=prob[:,j]
val_csv.to_csv(val_dir/'VALIDATION_PREDICTIONS.csv',index=False)
(val_dir/'VALIDATION_METRICS.json').write_text(json.dumps(metrics,indent=2),encoding='utf-8')
print('\nValidation artifacts:',val_dir)
print('Official TEST accessed: NO')
